In [ ]:
!pip install -q transformers sentence-transformers faiss-cpu langchain pypdf streamlit pyngrok langchain-community

In [ ]:
import os
import pickle
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from sentence_transformers import SentenceTransformer
from langchain.vectorstores import FAISS
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter


In [ ]:
from google.colab import files
uploaded = files.upload()  # This opens a file selector

# Move your PDF to `data/` folder
import os
os.makedirs("data", exist_ok=True)

for fn in uploaded.keys():
    os.rename(fn, f"data/{fn}")

!ls data/


Saving AI_Customer_Support_FAQ_Dataset.pdf to AI_Customer_Support_FAQ_Dataset.pdf
AI_Customer_Support_FAQ_Dataset.pdf


In [ ]:
!pip install -q transformers sentence-transformers faiss-cpu langchain langchain-community pypdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter

# Make sure your PDF filename matches exactly
pdf_path = "data/AI_Customer_Support_FAQ_Dataset.pdf"

# Load PDF
loader = PyPDFLoader(pdf_path)
pages = loader.load()

print(f"Loaded {len(pages)} pages from PDF.")
print("Preview first page:")
print(pages[0].page_content[:300])

# Split pages into chunks for embeddings
splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(pages)

print(f"Created {len(chunks)} chunks.")

Loaded 7 pages from PDF.
Preview first page:
AI-Powered Customer Support Chatbot FAQ
 Dataset (2025)
This dataset contains 100 frequently asked questions and answers used for RAG-based chatbot
training.
Q1: How do I reset my password?
A1: Click on 'Forgot Password' on the login page and follow the instructions sent to your email.
Q2: What is y
Created 7 chunks.


In [ ]:
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS

# Use a lightweight sentence-transformer model
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Create FAISS vector store
db = FAISS.from_documents(chunks, embeddings)

# Save index for later
db.save_local("faiss_index")
print("FAISS index saved locally.")


FAISS index saved locally.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

rag_pipeline = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    device=-1  # CPU, change to 0 for GPU if available
)


Device set to use cpu


In [ ]:
# Use the FAISS retriever
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k":3})

def answer_query(query):
    # Retrieve top 3 relevant chunks
    docs = retriever.get_relevant_documents(query)
    context = " ".join([doc.page_content for doc in docs])

    # Truncate context to avoid model overflow
    if len(context.split()) > 500:
        context = " ".join(context.split()[:500])

    # Prepare input for FLAN-T5
    input_text = f"Answer the question based on the context below:\nContext: {context}\nQuestion: {query}"

    output = rag_pipeline(input_text, max_new_tokens=256)
    return output[0]['generated_text']


In [ ]:
questions = [
    "I forgot my password, how do I reset it?",
    "Who can I contact for product returns?",
    "Explain the warranty period for product X.",
    "Do you have an office in Germany?",
    "What is the latest model of your product line?"
]

for q in questions:
    print(f"Q: {q}")
    print("A:", answer_query(q))
    print("------")


Token indices sequence length is longer than the specified maximum sequence length for this model (662 > 512). Running this sequence through the model will result in indexing errors


Q: I forgot my password, how do I reset it?
A: Click on 'Forgot Password' on the login page and follow the instructions sent to your email.
------
Q: Who can I contact for product returns?
A: our support team through live chat or email for immediate assistance.
------
Q: Explain the warranty period for product X.
A: Q49: How do I resolve warranty related issues? A49: For any warranty related concerns, please reach out to our support team through live chat or email for immediate assistance. Q50: How do I resolve warranty related issues? A50: For any warranty related concerns, please reach out to our support team through live chat or email for immediate assistance. Q50: How do I resolve refunds related issues? A50: For any refund related concerns, please reach out to our support team through live chat or email for immediate assistance. Q50: How do I resolve delivery related issues? A50: For any warranty related concerns, please reach out to our support team through live chat or email for

In [ ]:
# chatbot.py
from langchain.vectorstores import FAISS
from langchain.embeddings import SentenceTransformerEmbeddings
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# Load FAISS index
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.load_local("faiss_index", embeddings)

# Load FLAN-T5
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
rag_pipeline = pipeline("text2text-generation", model=model, tokenizer=tokenizer, device=-1)

# Setup retriever
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k":3})

def answer_query(query):
    docs = retriever.get_relevant_documents(query)
    context = " ".join([doc.page_content for doc in docs])
    if len(context.split()) > 500:
        context = " ".join(context.split()[:500])
    input_text = f"Answer the question based on the context below:\nContext: {context}\nQuestion: {query}"
    output = rag_pipeline(input_text, max_new_tokens=256)
    return output[0]['generated_text']

Device set to use cpu


In [ ]:
from langchain.vectorstores import FAISS
from langchain.embeddings import SentenceTransformerEmbeddings
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# Load FAISS index
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.load_local("faiss_index", embeddings)

# Load FLAN-T5
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
rag_pipeline = pipeline("text2text-generation", model=model, tokenizer=tokenizer, device=-1)

# Setup retriever
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k":3})

def answer_query(query):
    docs = retriever.get_relevant_documents(query)
    context = " ".join([doc.page_content for doc in docs])

    # Prepare input for FLAN-T5
    input_text = f"Answer the question based on the context below:\nContext: {context}\nQuestion: {query}"

    # Truncate input_text based on tokenizer's max length
    max_length = tokenizer.model_max_length - 10 # Leave some buffer for output
    if len(tokenizer(input_text)['input_ids']) > max_length:
        input_text = tokenizer.decode(tokenizer(input_text)['input_ids'][:max_length], skip_special_tokens=True)


    output = rag_pipeline(input_text, max_new_tokens=256)
    return output[0]['generated_text']


questions = [
    "I forgot my password, how do I reset it?",
    "Who can I contact for product returns?",
    "Explain the warranty period for product X.",
    "Do you have an office in Germany?",
    "What is the latest model of your product line?"
]

for q in questions:
    print(f"Q: {q}")
    print("A:", answer_query(q))
    print("------")

Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (1628 > 512). Running this sequence through the model will result in indexing errors


Q: I forgot my password, how do I reset it?
A: reach out to our support team through live chat or email for immediate assistance.
------
Q: Who can I contact for product returns?
A: please reach out to our support team through live chat or email for immediate assistance.
------
Q: Explain the warranty period for product X.
A: For any delivery related concerns, please reach out to our support team through live chat or email for immediate assistance.
------
Q: Do you have an office in Germany?
A: Q13: How do I resolve notifications related issues?
------
Q: What is the latest model of your product line?
A: Q13: How do I resolve notifications related issues?
------


In [ ]:
!zip -r faiss_index.zip faiss_index


updating: faiss_index/ (stored 0%)
updating: faiss_index/index.faiss (deflated 7%)
updating: faiss_index/index.pkl (deflated 86%)


In [67]:
%%writefile app.py
import streamlit as st
from langchain.vectorstores import FAISS
from langchain.embeddings import SentenceTransformerEmbeddings
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# Load FAISS index
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.load_local("faiss_index", embeddings)

# Load FLAN-T5
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
# Using device='cpu' explicitly for compatibility
rag_pipeline = pipeline("text2text-generation", model=model, tokenizer=tokenizer, device='cpu')


# Setup retriever
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k":5}) # Increased k to 5

def answer_query(query):
    docs = retriever.get_relevant_documents(query)
    context = " ".join([doc.page_content for doc in docs])
    # Truncate context to avoid model overflow, using token count approximation
    # A more precise method would involve using the tokenizer
    max_context_tokens = 400 # Adjusted based on model's limit (512) and prompt overhead
    if len(context.split()) > max_context_tokens:
        context = " ".join(context.split()[:max_context_tokens])

    # Prepare input for FLAN-T5
    input_text = f"Answer the question based on the context below:\nContext: {context}\nQuestion: {query}"

    output = rag_pipeline(input_text, max_new_tokens=256)
    return output[0]['generated_text']

st.title("Customer Support Chatbot 🤖")
st.write("Ask me anything about your products or support.")

query = st.text_input("Your question:")

if query:
    answer = answer_query(query)
    st.write("**Answer:**", answer)

Overwriting app.py


In [68]:
!pip install -q streamlit pyngrok


In [69]:
!streamlit run app.py &>/dev/null &

In [70]:
# Install pyngrok if not already
!pip install -q pyngrok

from pyngrok import ngrok
import os

# Kill any existing ngrok tunnels
ngrok.kill()

# Set your ngrok authtoken
# Replace YOUR_AUTHTOKEN below with your actual token
NGROK_AUTH_TOKEN = "345ZMsksxJb90EKNqRVdk0B3RhF_x19BbwCfrqWtwgMtrgQ5" # Correct assignment - Replace with your actual token or use Colab Secrets
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Run Streamlit app in background
get_ipython().system_raw("streamlit run app.py &")

# Connect ngrok to Streamlit port
public_url = ngrok.connect(8501)
print("Streamlit is live at:", public_url)

Streamlit is live at: NgrokTunnel: "https://josephina-diatomaceous-kyrie.ngrok-free.dev" -> "http://localhost:8501"
